# __Project: Mandarin ASR with wav2vec 2.0 and CTC__

- This project follows the general workflow of the Hugging Face tutorial [Fine-Tuning XLS-R for Multilingual ASR with 🤗 Transformers](https://huggingface.co/blog/fine-tune-xlsr-wav2vec2). 

- The original tutorial has been adapted to the current Hugging Face Transformers API and to Mandarin speech recognition using the AISHELL-3 corpus.

- The goal of the project is to fine-tune a pretrained XLS-R model for Mandarin automatic speech recognition (ASR) using Connectionist Temporal Classification (CTC). Two output representations are investigated:

1. Chinese characters (Hanzi)
2. Tone-marked Pinyin syllables

- This notebook covers the first part of the project, including corpus exploration, transcription parsing, train/validation/test preparation, Hugging Face dataset generation, vocabulary construction, and input preprocessing.

- _AI-assisted development was used for code explanation, debugging, and API migration. Python version 3.13.14, Transformers version 5.12.1_

In [28]:
import torch
import torchcodec
from pathlib import Path
from datasets import Dataset, DatasetDict, Audio
import re
import csv

print(torch.__version__)
print(torchcodec.__version__)

!pip list | grep transformer

2.15.0
0.17.0
transformers            5.12.1


### __📊 Environmental Setups and Get Data form the Dataset__

In [ ]:
# get the project root and the data root
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "aishell3"

print(PROJECT_ROOT)
print(DATA_ROOT)

# root
# /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR
# /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/aishell3

/Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR
/Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/aishell3


#### Get Corresponding Hanzi and Pinyin from the transcript

- We want to build Hugging Face Dataset, including one Hanzi Dataset and Pinyin Dataset

In [7]:
HANZI_RE = re.compile(r"^[\u4e00-\u9fff]+$") # using utf-8 to distinguish the chinese characters Hanzi
PINYIN_RE = re.compile(r"^[a-zA-ZüÜ]+[1-5]?$") # ascii letters with optional tone numbers

# functions to decide whether input is Hanzi or Pinyin, added to the corresponding list
def is_hanzi(token: str) -> bool:
    return bool(HANZI_RE.fullmatch(token))

def is_pinyin(token: str) -> bool:
    return bool(PINYIN_RE.fullmatch(token))

def parse_aishell3_line(line: str):
    parts = line.strip().split()
    if not parts:
        return None

    wav_name = parts[0] # fist part is the wav name
    tokens = parts[1:] # starts here for the real tokens

    hanzi_chars = []
    pinyin_chars = []

    for tok in tokens:
        if is_hanzi(tok):
            hanzi_chars.append(tok)
        elif is_pinyin(tok):
            pinyin_chars.append(tok)
        else:
            raise ValueError(f"Unsupported token: {tok!r} in line: {line.rstrip()}")

    hanzi = "".join(hanzi_chars)
    pinyin = " ".join(pinyin_chars)

    return wav_name, hanzi, pinyin

#### Define the Dataset Generator

In [9]:
def aishell3_generator(content_file, wav_root):
    content_file = Path(content_file)
    wav_root = Path(wav_root)

    with content_file.open("r", encoding="utf-8") as f:
        for _, line in enumerate(f, start=1):
            if not line.strip():
                continue

            parsed = parse_aishell3_line(line)
            if parsed is None:
                continue

            wav_name, hanzi, pinyin = parsed

            speaker_id = Path(wav_name).stem[:7]
            audio_path = wav_root / speaker_id / wav_name

            if not audio_path.exists():
                raise FileNotFoundError(f"Audio Not Found: {audio_path}")

            yield {
                "id": Path(wav_name).stem,
                "speaker_id": speaker_id,
                "audio": str(audio_path.resolve()),
                "hanzi": hanzi,
                "pinyin": pinyin,
            }

In [ ]:
# apply the dataset generator to the two ogriginal dataset
train_dataset = Dataset.from_generator(
    aishell3_generator,
    gen_kwargs={
        "content_file": str(DATA_ROOT / "train" / "content.txt"), # get the train data from train set
        "wav_root": str(DATA_ROOT / "train" / "wav"),
    },
)

test_dataset = Dataset.from_generator(
    aishell3_generator,
    gen_kwargs={
        "content_file": str(DATA_ROOT / "test" / "content.txt"), #get the test data from the test set
        "wav_root": str(DATA_ROOT / "test" / "wav"),
    },
)

aishell_base = DatasetDict({
    "train": train_dataset,
    "test": test_dataset,
})

aishell_base

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin'],
        num_rows: 63262
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin'],
        num_rows: 24773
    })
})

In [ ]:
# cast the "audio" column to the Audio type, set sample rate to 16kHz and number of channels to 1 (mono)
aishell_base = aishell_base.cast_column("audio", Audio(sampling_rate=16_000, num_channels=1),)

In [ ]:
# sanity check I: if the generator works well on the dataset
samples = [aishell_base["train"][0], aishell_base["test"][0],]

for sample in samples:
    print("ID:", sample["id"])
    print("Speaker:", sample["speaker_id"])
    print("Hanzi:", sample["hanzi"])
    print(f"Pinyin:", sample["pinyin"], "\n")


ID: SSB00050001
Speaker: SSB0005
Hanzi: 广州女大学生登山失联四天警方找到疑似女尸
Pinyin: guang3 zhou1 nv3 da4 xue2 sheng1 deng1 shan1 shi1 lian2 si4 tian1 jing3 fang1 zhao3 dao4 yi2 si4 nv3 shi1 

ID: SSB06930002
Speaker: SSB0693
Hanzi: 武术始终被看作我国的国粹
Pinyin: wu3 shu4 shi3 zhong1 bei4 kan4 zuo4 wo3 guo2 de5 guo2 cui4 



In [26]:
# sanity check II: Sample rate and audio length of the audio data.
audio = sample["audio"]

print(audio)
print("sample_rate:", audio["sampling_rate"])
print("audio_length:", len(audio["array"]))

sample_rate: 16000
audio_length: 52336


#### Get Speaker Metadata

In [29]:
metadata_path = DATA_ROOT/"spk-info.txt"

speaker_metadata = {}

with metadata_path.open("r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")

    for line_number, row in enumerate(reader, start=1):

        if not row or not any(value.strip() for value in row):
            continue

        if row[0].strip().startswith("#"):
            continue

        row = [value.strip() for value in row]

        if len(row) != 4:
            raise ValueError(
                f"Invalid metadata row {line_number}: "
                f"expected 4 columns, got {len(row)}\n"
                f"Row: {row}"
            )

        speaker_id, age_group, gender, accent = row

        speaker_id = speaker_id.upper()

        if age_group not in {"A", "B", "C", "D"}:
            raise ValueError(
                f"Unknown age group '{age_group}' "
                f"for speaker {speaker_id}"
            )

        if speaker_id in speaker_metadata:
            raise ValueError(f"Duplicate metadata for speaker: {speaker_id}")

        speaker_metadata[speaker_id] = {
            "age_group": age_group,
            "gender": gender.lower(),
            "accent": accent.lower()
        }

print("Speakers in metadata:", len(speaker_metadata))

Speakers in metadata: 218


In [ ]:
# sanity check: If the number of the total speaker in metadata match
dataset_speakers = set()

for split_dataset in aishell_base.values():
    dataset_speakers.update(split_dataset.unique("speaker_id"))

metadata_speakers = set(speaker_metadata)
missing_metadata = dataset_speakers - metadata_speakers
unused_metadata = metadata_speakers - dataset_speakers

print("Speakers in dataset:", len(dataset_speakers))
print("Speakers without metadata:", sorted(missing_metadata))
print("Unused metadata speakers:", len(unused_metadata))

if missing_metadata:
    raise ValueError(
        f"{len(missing_metadata)} speakers have no metadata: "
        f"{sorted(missing_metadata)}"
    )

Speakers in dataset: 218
Speakers without metadata: []
Unused metadata speakers: 0


In [ ]:
# enrich each dataset sample with speaker metadata, find the corresponding metadata first
def add_speaker_metadata(speaker_id):
    speaker_id = speaker_id.upper()

    if speaker_id not in speaker_metadata:
        raise ValueError(
            f"Missing metadata for speaker: {speaker_id}"
        )
    return speaker_metadata[speaker_id]

aishell_base = aishell_base.map(
    add_speaker_metadata,
    input_columns="speaker_id",
    desc="Adding speaker metadata",
)

print(aishell_base["train"][0])

Adding speaker metadata:   0%|          | 0/63262 [00:00<?, ? examples/s]

Adding speaker metadata:   0%|          | 0/24773 [00:00<?, ? examples/s]

{'id': 'SSB00050001', 'speaker_id': 'SSB0005', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x30ce3dff0>, 'hanzi': '广州女大学生登山失联四天警方找到疑似女尸', 'pinyin': 'guang3 zhou1 nv3 da4 xue2 sheng1 deng1 shan1 shi1 lian2 si4 tian1 jing3 fang1 zhao3 dao4 yi2 si4 nv3 shi1', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}


#### Split the Dataset into Train, Validation, and Test Sets

- We already have the train and test splits, but we need to create a validation split. We will use 10% of the training data for validation.

❗️ We noticed that there are few issues with the dataset splits. The issues include:

- The overlap of speakers in train and test set.
- The imbalance of the data amount between the train and test set.

These issues are from the original dataset, so we decided to keep following the split of the original set and not to edit it.

In [ ]:
# split 10% of the original training set as validation set
train_validation_split = aishell_base["train"].train_test_split(
    test_size=0.1,
    seed=42,
    shuffle=True,
)

aishell_base = DatasetDict({
    "train": train_validation_split["train"],
    "validation": train_validation_split["test"],
    "test": aishell_base["test"],
})

print(aishell_base)
print("Train:", len(aishell_base["train"]))
print("Validation:", len(aishell_base["validation"]))
print("Test:", len(aishell_base["test"]))

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
Train: 56935
Validation: 6327
Test: 24773


### __📚 Get the Hanzi and Pinyin Dataset, Build and Store the Dataset__

In [33]:
# remove the "pinyin" column and rename "hanzi" to "text"
hanzi_dataset = aishell_base.remove_columns(["pinyin"])
hanzi_dataset = hanzi_dataset.rename_column("hanzi", "text")

print(hanzi_dataset)
print(hanzi_dataset["train"][0])
print(hanzi_dataset["validation"][0])
print(hanzi_dataset["test"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
{'id': 'SSB03820023', 'speaker_id': 'SSB0382', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x30d087cf0>, 'text': '但这一点已经足够表明卷福的超高人气', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}
{'id': 'SSB03410187', 'speaker_id': 'SSB0341', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x30d087cf0>, 'text': '中联航将被削减百分之十的飞行小时数', 'age_group': 'C', 'gender': 'female', 'accent': 'north'}
{'id': 'SSB06930002', 'speaker_id': 'SSB0693', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x128a04170>, 'text': '武

In [34]:
# remove the "hanzi" column and rename "pinyin" to "text"
pinyin_dataset = aishell_base.remove_columns(["hanzi"])
pinyin_dataset = pinyin_dataset.rename_column("pinyin", "text")

print(pinyin_dataset)
print(pinyin_dataset["train"][0])
print(pinyin_dataset["validation"][0])
print(pinyin_dataset["test"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
{'id': 'SSB03820023', 'speaker_id': 'SSB0382', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x128a4fcd0>, 'text': 'dan4 zhe4 yi4 dian2 yi3 jing1 zu2 gou4 biao3 ming2 juan4 fu2 de5 chao1 gao1 ren2 qi4', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}
{'id': 'SSB03410187', 'speaker_id': 'SSB0341', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x128a4e150>, 'text': 'zhong1 lian2 hang2 jiang1 bei4 xiao1 jian3 bai3 fen1 zhi1 shi2 de5 fei1 xing2 xiao3 shi2 shu4', 'age_group': 'C', 'gender': 'female', 'accent': 

In [ ]:
# sanity check I: if all the id pass, if there're data left
for split in ["train", "validation", "test"]:
    hanzi_ids = hanzi_dataset[split]["id"]
    pinyin_ids = pinyin_dataset[split]["id"]

    assert hanzi_ids == pinyin_ids

print("Hanzi and Pinyin datasets use identical splits.")

Hanzi and Pinyin datasets use identical splits.


In [ ]:
# sanity check II: if there are utterance overlap
# Although the speakers are overlapped, the utt id should not be overlapped!

train_ids = set(hanzi_dataset["train"]["id"])
validation_ids = set(hanzi_dataset["validation"]["id"])
test_ids = set(hanzi_dataset["test"]["id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

print("No utterance overlap between train, validation, and test.")

No utterance overlap between train, validation, and test.


In [ ]:
# save dataset
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True,)

hanzi_dataset_path = (PROCESSED_DATA_DIR / "aishell3_hanzi")
pinyin_dataset_path = (PROCESSED_DATA_DIR / "aishell3_pinyin")

hanzi_dataset.save_to_disk(hanzi_dataset_path)

pinyin_dataset.save_to_disk(pinyin_dataset_path)

print("Saved Hanzi dataset:", hanzi_dataset_path.resolve())
print("Saved Pinyin dataset:", pinyin_dataset_path.resolve())

Saving the dataset (0/36 shards):   0%|          | 0/56935 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/6327 [00:00<?, ? examples/s]

Saving the dataset (0/15 shards):   0%|          | 0/24773 [00:00<?, ? examples/s]

Saving the dataset (0/36 shards):   0%|          | 0/56935 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/6327 [00:00<?, ? examples/s]

Saving the dataset (0/15 shards):   0%|          | 0/24773 [00:00<?, ? examples/s]

Saved Hanzi dataset: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processed_data/aishell3_hanzi
Saved Pinyin dataset: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processed_data/aishell3_pinyin
